# Mode 2 — In-process local vLLM

[English guide](../README.md) · [中文说明](../README.zh-CN.md)

This notebook loads the final merged 2B model locally and calls vLLM's offline API. It starts no HTTP server and uses no cloud key, Analyze, examples, or LoRA at runtime.

Prerequisites: Linux/NVIDIA GPU, downloaded final merged weights, Python 3.12 kernel; from the repository root install `pip install -r requirements-notebook.txt`. The model download URL is pending; see `docs/MODELS.md`.

**Run only on an idle GPU.** Set `ORDO_MODEL_PATH` and choose `CUDA_VISIBLE_DEVICES` before importing torch/vLLM. Default 0.45 GPU budget was validated on one 24GB RTX 3090, not every machine. Restart the kernel to release the engine/GPU when finished. No notebook cells were executed in the release preparation.

In [ ]:
import os
from pathlib import Path
import sys

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')  # Change to an idle physical GPU.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'local_service' / 'ordo_local').is_dir())
sys.path.insert(0, str(ROOT / 'local_service'))
MODEL = Path(os.environ.get('ORDO_MODEL_PATH', str(ROOT / 'models' / 'Ordo-Qwen3.5-2B-Reviewed2-Final-BF16'))).expanduser().resolve()
assert (MODEL / 'config.json').is_file(), 'Download final merged weights first; see docs/MODELS.md'


## Load the model

This cell allocates GPU memory. Model loading/compilation is separate from document inference time. Do not run this while the HTTP deployment already occupies the same GPU.

In [ ]:
from ordo_local.offline import load_model, label_text
from ordo_local.client import save_result

llm = load_model(MODEL, gpu_memory_utilization=0.45, max_model_len=8192)


## Label text and export selected formats

Replace the sample with any UTF-8 TXT input. Empty lines are not labeled. Original nonempty text is preserved in JSON/TXT; Markdown adds structure. Eighteen current lines are labeled per window, with previous/next raw context.

In [ ]:
text = (ROOT / 'examples' / 'sample.txt').read_text(encoding='utf-8-sig')
result = label_text(llm, text, formats=['json', 'txt', 'md'])
print(result['meta'])
print(next(f['content'] for f in result['files'] if f['filename'] == 'labeled.txt'))


In [ ]:
# Refuses to overwrite an existing result: choose a new directory for another run.
save_result(result, ROOT / 'outputs' / 'notebook_sample')


For repeated calls, keep `llm` loaded. To serve other processes, use Mode 3; to expose a text-upload business API, use Mode 4. See the repository README. These modes should not be launched on the same GPU at the same time unless you explicitly budget memory.